# NB_CursorOnboard_Gold
**Source:** `AI_Medallion.Silver.cursor_usage`  
**Target:** `AI_Medallion.Gold.fact_cursor_onboard`  
**Grain:** One row per `OktaUserId` per `CursorUsageDate` (latest ingestion)  
**Load type:** Incremental merge on `OnboardSurrogateKey`

In [ ]:
import os
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_LOCAL_TABLE_CACHE", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "75000")
os.environ.setdefault("MEDALLION_SCAN_RETRIES", "6")
# After kernel restart: run this cell, bootstrap, then read/transform/write cells.
# read_table() caches parquet under ~/.fabric/medallion_cache/ (Windows: %USERPROFILE%\.fabric\medallion_cache\).
# Re-read from OneLake (ignore cache): os.environ["MEDALLION_CACHE_REFRESH"] = "1"
# Delete ALL local cache: remove the medallion_cache folder (PowerShell: Remove-Item -Recurse -Force $env:USERPROFILE\.fabric\medallion_cache)
# Delete ONE table cache: remove its subfolder (e.g. AI_Medallion__Silver__cursor_usage).
# Disable caching: os.environ["MEDALLION_LOCAL_TABLE_CACHE"] = "0"


In [ ]:
%run ../common/bootstrap


In [ ]:
silver = read_table_broken_lineage(TABLE_CURSOR_SILVER, "cursor_silver")


In [ ]:
apply_local_merge_spark_conf(spark)

daily = break_lineage_local(
    cursor_onboard_daily_dedupe(silver), "cursor_onboard_daily"
)
daily = daily.withColumn(FACT_CURSOR_ONBOARD_MERGE_KEY, cursor_onboard_merge_key())

onboard = daily.select(
    F.lit(None).cast("string").alias("account"),
    F.lit(None).cast("string").alias("projectid"),
    F.lit(None).cast("string").alias("projectName"),
    F.col("OktaUserFullName").alias("username"),
    F.col("OktaUserEmployeeNumber").alias("userNumberEmployee"),
    F.col("OktaUserEmail").alias("userEmail"),
    F.col("OktaUserFullEmail").alias("userFullEmail"),
    F.col("OktaUserRegion").alias("userRegion"),
    F.col("OktaUserCountry").alias("userCountry"),
    F.col("CursorUsageDate").alias("LastModification"),
    F.col("HCBusinessUnit").alias("Employee_Business_Unit"),
    F.col("HCJobTitle").alias("Job_Roles"),
    F.col("OktaUserCountry").alias("Employee_Country"),
    F.col("HCOffice").alias("Employee_Office"),
    F.col("HCTalentSegment").alias("Employee_Segment"),
    F.col("HCDivision").alias("Foundation"),
    F.col("HCBusinessUnitCode").alias("Employee_Business_Unit_Code"),
    F.lit(None).cast("string").alias("billingClientName"),
    F.lit(None).cast("string").alias("billingCountry"),
    F.lit(None).cast("string").alias("billingRegion"),
    F.col(FACT_CURSOR_ONBOARD_MERGE_KEY),
)

onboard = onboard.filter(
    F.col("userEmail").isNotNull()
    & F.col("userFullEmail").isNotNull()
    & F.col("LastModification").isNotNull()
    & F.col(FACT_CURSOR_ONBOARD_MERGE_KEY).isNotNull()
)

publish_merge_staging(onboard, TABLE_FACT_CURSOR_ONBOARD)


In [ ]:
merge_incremental(
    spark,
    onboard,
    TABLE_FACT_CURSOR_ONBOARD,
    FACT_CURSOR_ONBOARD_MERGE_KEY,
)

show_sample(onboard.limit(10))
